In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler

In [2]:
try:
    import shap
    print("[INFO] SHAP library found")
except ImportError:
    shap = None
    print("[INFO] SHAP library not found, skipping SHAP-related features")

print("All libraries imported successfully")

[INFO] SHAP library found
All libraries imported successfully


In [4]:
df = pd.read_csv("Afficionado Coffee Roasters.xlsx - Transactions.csv")
df.info()
df.shape

<class 'pandas.DataFrame'>
RangeIndex: 149116 entries, 0 to 149115
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   transaction_id    149116 non-null  int64  
 1   year              149116 non-null  int64  
 2   transaction_time  149116 non-null  str    
 3   transaction_qty   149116 non-null  int64  
 4   store_id          149116 non-null  int64  
 5   store_location    149116 non-null  str    
 6   product_id        149116 non-null  int64  
 7   unit_price        149116 non-null  float64
 8   product_category  149116 non-null  str    
 9   product_type      149116 non-null  str    
 10  product_detail    149116 non-null  str    
dtypes: float64(1), int64(5), str(5)
memory usage: 20.9 MB


(149116, 11)

# ---Data Ingestion and Validation---

In [5]:
missing_values = df.isnull().sum().sum()
duplicate_ids = df['transaction_id'].duplicated().sum()
invalid_qty = (df['transaction_qty'] <= 0).sum()
invalid_price = (df['unit_price'] <= 0).sum()

In [6]:
print("--- DATA VALIDATION ---")
print(f"Missing Values: {missing_values}")
print(f"Duplicate Transaction IDs: {duplicate_ids}")
print(f"Non-positive Quantities: {invalid_qty}")
print(f"Non-positive Prices: {invalid_price}")

--- DATA VALIDATION ---
Missing Values: 0
Duplicate Transaction IDs: 0
Non-positive Quantities: 0
Non-positive Prices: 0


In [7]:
df = df.sort_values('transaction_id').reset_index(drop=True)
time_seconds = pd.to_timedelta(df['transaction_time']).dt.total_seconds()
time_diff = time_seconds.diff()

In [8]:
day_transitions = (time_diff < 0).sum()
print(f"\n--- TIME ANALYSIS ---")
print(f"Number of day transitions detected (time resetting): {day_transitions}")
print(f"Total implied days: {day_transitions + 1}")


--- TIME ANALYSIS ---
Number of day transitions detected (time resetting): 180
Total implied days: 181


In [9]:
# Reconstruct the Date column
start_date = pd.to_datetime('2025-01-01')
day_increments = (time_diff < 0).cumsum().fillna(0)
df['transaction_date'] = start_date + pd.to_timedelta(day_increments, unit='D')

In [10]:
# --- FEATURE ENGINEERING (TEMPORAL) ---
# 1. Revenue per transaction
df['revenue'] = df['transaction_qty'] * df['unit_price']

# 2. Hour of day (0-23)
df['hour'] = pd.to_datetime(df['transaction_time'], format='%H:%M:%S').dt.hour

# 3. Day of week (Monday–Sunday)
df['day_of_week'] = df['transaction_date'].dt.day_name()

In [11]:
# 4. Time buckets
def assign_time_bucket(hour):
    if 6 <= hour <= 11:
        return 'Morning (6-11)'
    elif 12 <= hour <= 16:
        return 'Afternoon (12-16)'
    elif 17 <= hour <= 21:
        return 'Evening (17-21)'
    else:
        return 'Late hours (22-5)'

df['time_bucket'] = df['hour'].apply(assign_time_bucket)

# Rearrange columns slightly for better readability
cols = ['transaction_id', 'transaction_date', 'day_of_week', 'transaction_time', 'hour', 'time_bucket', 
        'transaction_qty', 'unit_price', 'revenue', 'store_id', 'store_location', 
        'product_id', 'product_category', 'product_type', 'product_detail']

df_final = df[cols]

# Save the final preprocessed data
output_file = 'Afficionado_Coffee_Processed.csv'
df_final.to_csv(output_file, index=False)

In [12]:
print(f"\nSuccessfully created {output_file}")
print("--- PREPROCESSED DATA SAMPLE ---")
print(df_final[['transaction_date', 'day_of_week', 'transaction_time', 'time_bucket', 'revenue']].head(10))
print("\n--- DISTRIBUTION OF TIME BUCKETS ---")
print(df_final['time_bucket'].value_counts())


Successfully created Afficionado_Coffee_Processed.csv
--- PREPROCESSED DATA SAMPLE ---
  transaction_date day_of_week transaction_time     time_bucket  revenue
0       2025-01-01   Wednesday          7:06:11  Morning (6-11)     6.00
1       2025-01-01   Wednesday          7:08:56  Morning (6-11)     6.20
2       2025-01-01   Wednesday          7:14:04  Morning (6-11)     9.00
3       2025-01-01   Wednesday          7:20:24  Morning (6-11)     2.00
4       2025-01-01   Wednesday          7:22:41  Morning (6-11)     6.20
5       2025-01-01   Wednesday          7:22:41  Morning (6-11)     3.00
6       2025-01-01   Wednesday          7:25:49  Morning (6-11)     2.00
7       2025-01-01   Wednesday          7:33:34  Morning (6-11)     4.00
8       2025-01-01   Wednesday          7:39:13  Morning (6-11)     4.25
9       2025-01-01   Wednesday          7:39:34  Morning (6-11)     7.00

--- DISTRIBUTION OF TIME BUCKETS ---
time_bucket
Morning (6-11)       81751
Afternoon (12-16)    44427
Eveni

In [13]:
df.info()
df.describe()


<class 'pandas.DataFrame'>
RangeIndex: 149116 entries, 0 to 149115
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   transaction_id    149116 non-null  int64         
 1   year              149116 non-null  int64         
 2   transaction_time  149116 non-null  str           
 3   transaction_qty   149116 non-null  int64         
 4   store_id          149116 non-null  int64         
 5   store_location    149116 non-null  str           
 6   product_id        149116 non-null  int64         
 7   unit_price        149116 non-null  float64       
 8   product_category  149116 non-null  str           
 9   product_type      149116 non-null  str           
 10  product_detail    149116 non-null  str           
 11  transaction_date  149116 non-null  datetime64[us]
 12  revenue           149116 non-null  float64       
 13  hour              149116 non-null  int32         
 14  day_of_week    

,transaction_id,year,transaction_qty,store_id,product_id,unit_price,transaction_date,revenue,hour
count,149116.000000,149116.0,149116.000000,149116.000000,149116.000000,149116.000000,149116,149116.000000,149116.000000
mean,74737.371872,2025.0,1.438276,5.342063,47.918607,3.382219,2025-04-15 11:50:32.173609,4.686367,11.735790
min,1.000000,2025.0,1.000000,3.000000,1.000000,0.800000,2025-01-01 00:00:00,0.800000,6.000000
25%,37335.750000,2025.0,1.000000,3.000000,33.000000,2.500000,2025-03-06 00:00:00,3.000000,9.000000
50%,74727.500000,2025.0,1.000000,5.000000,47.000000,3.000000,2025-04-24 00:00:00,3.750000,11.000000
75%,112094.250000,2025.0,2.000000,8.000000,60.000000,3.750000,2025-05-30 00:00:00,6.000000,15.000000
max,149456.000000,2025.0,8.000000,8.000000,87.000000,45.000000,2025-06-30 00:00:00,360.000000,20.000000
std,43153.600016,0.0,0.542509,2.074241,17.930020,2.658723,NaN,4.227099,3.764662
